In [ ]:
import mne, numpy as np, json
from pathlib import Path

ROOT = Path("..").resolve()
EDF = ROOT / "data" / "SC4001E0-PSG.edf"

raw = mne.io.read_raw_edf(EDF, preload=False, verbose="ERROR")

print("채널 수:", len(raw.ch_names))
print("전체 샘플 수:", raw.n_times)
print("기록 길이(초):", raw.n_times / raw.info["sfreq"])
print("측정 시작:", raw.info["meas_date"])
print()
for i, ch in enumerate(raw.ch_names):
    print(f"[{i}] {ch}")

## 정답지(fixture) 생성

MNE로 파일을 읽어, 각 채널의 처음 300초 구간에 대해 디지털 정수값과 물리값(uV/DegC 등)을
`src/test/resources/expected_ch{N}_digital.csv`, `expected_ch{N}_physical.csv`로 저장한다.
Java의 `EdfReaderTest`가 이 파일들과 1e-6 오차 이내로 일치하는지 검증한다.

**주의**: EDF 파일은 채널마다 샘플링 주파수가 다르다(EEG/EOG 100Hz, 나머지 1Hz). MNE은 내부적으로
모든 채널을 최고 샘플링 주파수(`raw.info["sfreq"]`)에 맞춰 저장하므로, 저속 채널을 그냥
`raw[ch, 0:n]`으로 슬라이싱하면 실제로는 업샘플된 값을 받게 된다. 반드시 내부 sfreq 기준으로
슬라이싱한 뒤, `내부sfreq/채널고유sfreq` 간격으로 다운샘플링해서 원래 샘플만 추출해야 한다.
(실제로 이 단계를 빠뜨렸다가 저속 채널 정답지가 보간된 값으로 잘못 생성된 적이 있다.)


In [ ]:
WINDOW_SEC = 300.0
OUT = ROOT / "src" / "test" / "resources"

raw = mne.io.read_raw_edf(EDF, preload=True, verbose="ERROR")  # preload=True 필수 (아래 설명 참고)
extras = raw._raw_extras[0]
cal = extras["cal"]
offsets = extras["offsets"]
units = extras["units"]

n_samps = extras["n_samps"]
record_length = extras["record_length"]
if isinstance(record_length, (tuple, list, np.ndarray)):
    record_length = record_length[0]
sfreq_per_ch = n_samps / record_length

internal_sfreq = raw.info["sfreq"]  # 모든 채널이 내부적으로 이 주파수로 저장됨

for ch in range(len(raw.ch_names)):
    sfreq = sfreq_per_ch[ch]
    n_internal = int(round(WINDOW_SEC * internal_sfreq))
    data, _ = raw[ch, 0:n_internal]

    stride = int(round(internal_sfreq / sfreq))
    native = data[0][::stride] / units[ch]
    digital = np.round((native - offsets[ch]) / cal[ch]).astype(np.int64)

    (OUT / f"expected_ch{ch}_digital.csv").write_text("\n".join(str(v) for v in digital) + "\n")
    (OUT / f"expected_ch{ch}_physical.csv").write_text("\n".join(f"{v:.9f}" for v in native) + "\n")

    print(f"[{ch}] {raw.ch_names[ch]}: sfreq={sfreq}Hz, {len(digital)}개 샘플 저장")


## 노이즈 제거 필터 정답지

Java 서버가 실시간으로 적용하는 대역통과(0.5–40Hz, 4차 Butterworth) 필터가 MNE와 정확히
같은 결과를 내는지 검증하기 위한 정답지. `raw.filter(method='iir', phase='forward')`를 쓰면
scipy의 `butter()` + `lfilter()`(인과적/1방향 적용)와 완전히 동일해서, 실시간 스트리밍처럼
"미래 데이터 없이 지금까지 들어온 데이터만으로" 필터링하는 상황과 맞아떨어진다.
(반대로 기본값인 `phase='zero'`는 순방향+역방향으로 두 번 적용하는 filtfilt라 미래 데이터가
필요해서 실시간 스트리밍에는 쓸 수 없다.)

100Hz로 샘플링된 채널 기준으로 대역통과 상한 40Hz는 Nyquist(50Hz) 안에 잘 들어오지만,
50/60Hz 노치 필터는 Nyquist에 걸쳐서 이 데이터에는 적용할 수 없다 — 그래서 이번에는
대역통과만 구현했다.


In [ ]:
FILTER_WINDOW_SEC = 60.0
ch_idx = 0
ch_name = raw.ch_names[ch_idx]
sfreq = raw.info["sfreq"]
n = int(round(FILTER_WINDOW_SEC * sfreq))

raw_filtered = raw.copy().filter(
    l_freq=0.5, h_freq=40, picks=[ch_name], method="iir", phase="forward", verbose="ERROR"
)
data_filt = raw_filtered.get_data(picks=[ch_name])[0][:n] * 1e6  # V -> uV

path = OUT / f"expected_ch{ch_idx}_filtered.csv"
path.write_text("\n".join(f"{v:.9f}" for v in data_filt) + "\n")
print(f"[{ch_idx}] {ch_name}: {len(data_filt)}개 필터링된 샘플 저장 -> {path.name}")
